In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import Ridge, Lasso

from sklearn.metrics import mean_squared_error, mean_absolute_error

In [3]:
data = pd.read_csv("data.csv")   # your file name
print(data.head())
print(data.shape)

  area\tbedrooms\tbathrooms\tfloor\tage\tdistance_to_metro\tschool_proximity\tprice
0  1398.685661\t5\t2\t18\t16\t14.07\t7.9\t114.670...                               
1    1144.69428\t4\t1\t3\t3\t6.14\t0.37\t126.2058733                               
2  1459.075415\t3\t3\t6\t11\t8.57\t0.66\t147.0826419                               
3  1809.211943\t2\t1\t3\t12\t11.88\t0.76\t164.729...                               
4  1106.33865\t2\t3\t24\t3\t10.49\t7.34\t99.69918747                               
(3800, 1)


In [5]:
# Z-score calculation
#data['z_score'] = (data['area'] - data['area'].mean()) / data['area'].std()

# Outliers where |Z| > 3
outliers = data[np.abs(data['z_score']) > 3]

print("Number of outliers:", len(outliers))

# Remove outliers
data_clean = data[np.abs(data['z_score']) <= 3]

print("Clean dataset size:", data_clean.shape)

KeyError: 'z_score'

In [ ]:
corr_matrix = data_clean.corr()
print(corr_matrix)

# Find highly correlated pairs (>0.8)
for i in corr_matrix.columns:
    for j in corr_matrix.columns:
        if i != j and abs(corr_matrix.loc[i, j]) > 0.8:
            print(i, "-", j, ":", corr_matrix.loc[i, j])

In [ ]:
X = data_clean.drop(['price', 'z_score'], axis=1)
y = data_clean['price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = MinMaxScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)

print("Coefficients:", ridge.coef_)

In [ ]:
# Largest positive & negative
features = X.columns

coef_df = pd.DataFrame({
    'Feature': features,
    'Coefficient': ridge.coef_
})

print(coef_df.sort_values(by='Coefficient', ascending=False))

In [ ]:
y_pred = ridge.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

print("RMSE:", rmse)
print("MAE:", mae)
print("MAPE:", mape)

In [ ]:
alphas = [0.01, 0.1, 1, 10, 100]

for a in alphas:
    model = Ridge(alpha=a)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    print("Alpha:", a, "RMSE:", rmse)

In [ ]:
test_df = pd.DataFrame({
    'actual': y_test,
    'pred': y_pred
})

# Buckets
low = test_df[test_df['actual'] < 50]
medium = test_df[(test_df['actual'] >= 50) & (test_df['actual'] <= 150)]
high = test_df[test_df['actual'] > 150]

def calc_rmse(df):
    return np.sqrt(mean_squared_error(df['actual'], df['pred']))

print("Low RMSE:", calc_rmse(low))
print("Medium RMSE:", calc_rmse(medium))
print("High RMSE:", calc_rmse(high))

In [ ]:
best_alpha = 1   # replace with best from above

ridge = Ridge(alpha=best_alpha)
lasso = Lasso(alpha=best_alpha)

ridge.fit(X_train, y_train)
lasso.fit(X_train, y_train)

ridge_pred = ridge.predict(X_test)
lasso_pred = lasso.predict(X_test)

ridge_rmse = np.sqrt(mean_squared_error(y_test, ridge_pred))
lasso_rmse = np.sqrt(mean_squared_error(y_test, lasso_pred))

print("Ridge RMSE:", ridge_rmse)
print("Lasso RMSE:", lasso_rmse)

# Non-zero coefficients
ridge_nonzero = np.sum(ridge.coef_ != 0)
lasso_nonzero = np.sum(lasso.coef_ != 0)

print("Ridge non-zero:", ridge_nonzero)
print("Lasso non-zero:", lasso_nonzero)